# Toronto Traffic Collision Analysis
Police Annual Statistical Report — 2014 to 2026.  
~809 000 records covering every reported collision in the city.

In [1]:
from pathlib import Path

import altair as alt
import polars as pl

## 1. Load & Schema

In [2]:
RAW = Path(
    "../data/raw/police-annual-statistical-report-traffic-collisions/traffic-collisions-4326.csv"
)

df = pl.read_csv(
    RAW,
    schema_overrides={
        "FATALITIES": pl.Utf8,
        "HOOD_158": pl.Utf8,
    },
    null_values=["NSA", "None", ""],
    infer_schema_length=10_000,
)

print(f"Rows: {len(df):,}   Columns: {df.width}")
df.schema

Rows: 809,034   Columns: 21


Schema([('_id', Int64),
        ('OCC_DATE', Int64),
        ('OCC_MONTH', String),
        ('OCC_DOW', String),
        ('OCC_YEAR', Int64),
        ('OCC_HOUR', Int64),
        ('DIVISION', String),
        ('FATALITIES', String),
        ('INJURY_COLLISIONS', String),
        ('FTR_COLLISIONS', String),
        ('PD_COLLISIONS', String),
        ('HOOD_158', String),
        ('NEIGHBOURHOOD_158', String),
        ('LONG_WGS84', Float64),
        ('LAT_WGS84', Float64),
        ('AUTOMOBILE', String),
        ('MOTORCYCLE', String),
        ('PASSENGER', String),
        ('BICYCLE', String),
        ('PEDESTRIAN', String),
        ('geometry', String)])

In [3]:
bool_cols = [
    "INJURY_COLLISIONS",
    "FTR_COLLISIONS",
    "PD_COLLISIONS",
    "AUTOMOBILE",
    "MOTORCYCLE",
    "PASSENGER",
    "BICYCLE",
    "PEDESTRIAN",
]

df = df.with_columns([pl.col(c).eq("YES").alias(c) for c in bool_cols]).with_columns(
    pl.col("FATALITIES").cast(pl.Int32).fill_null(0).alias("FATALITIES"),
    pl.col("OCC_YEAR").cast(pl.Int32),
    pl.col("OCC_HOUR").cast(pl.Int32),
)

df_geo = df.filter((pl.col("LAT_WGS84") != 0) & (pl.col("LONG_WGS84") != 0))
print(f"Records with valid coordinates: {len(df_geo):,} ({len(df_geo) / len(df):.1%})")
df.head(3)

Records with valid coordinates: 677,056 (83.7%)


_id,OCC_DATE,OCC_MONTH,OCC_DOW,OCC_YEAR,OCC_HOUR,DIVISION,FATALITIES,INJURY_COLLISIONS,FTR_COLLISIONS,PD_COLLISIONS,HOOD_158,NEIGHBOURHOOD_158,LONG_WGS84,LAT_WGS84,AUTOMOBILE,MOTORCYCLE,PASSENGER,BICYCLE,PEDESTRIAN,geometry
i64,i64,str,str,i32,i32,str,i32,bool,bool,bool,str,str,f64,f64,bool,bool,bool,bool,bool,str
1,1388552400000,"""January""","""Wednesday""",2014,1,"""D52""",0,false,false,true,"""168""","""Downtown Yonge East (168)""",-79.378428,43.65041,true,false,false,false,false,"""{""coordinates"": [[-79.37842774…"
2,1388552400000,"""January""","""Wednesday""",2014,22,"""D32""",0,true,false,false,null,null,0.0,0.0,true,false,false,false,false,"""{""coordinates"": [[5.6843418860…"
3,1388552400000,"""January""","""Wednesday""",2014,2,null,0,true,false,false,null,null,0.0,0.0,true,false,false,false,false,"""{""coordinates"": [[5.6843418860…"


In [4]:
summary = pl.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(d) for d in df.dtypes],
        "non_null": [df[c].drop_nulls().len() for c in df.columns],
        "null_pct": [round(df[c].null_count() / len(df) * 100, 1) for c in df.columns],
    }
)
summary

column,dtype,non_null,null_pct
str,str,i64,f64
"""_id""","""Int64""",809034,0.0
"""OCC_DATE""","""Int64""",809034,0.0
"""OCC_MONTH""","""String""",809034,0.0
"""OCC_DOW""","""String""",809034,0.0
"""OCC_YEAR""","""Int32""",809034,0.0
…,…,…,…
"""MOTORCYCLE""","""Boolean""",809030,0.0
"""PASSENGER""","""Boolean""",809030,0.0
"""BICYCLE""","""Boolean""",809030,0.0


## 2. Monthly Collision Trends

In [5]:
MONTH_IDX = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}

by_month_ts = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .with_columns(
        (
            pl.col("OCC_YEAR").cast(pl.Utf8)
            + "-"
            + pl.col("OCC_MONTH").map_elements(
                lambda m: str(MONTH_IDX[m]).zfill(2), return_dtype=pl.Utf8
            )
        ).alias("year_month")
    )
    .group_by("year_month")
    .agg(
        pl.len().alias("total_collisions"),
        pl.col("FATALITIES").sum().alias("fatalities"),
        pl.col("INJURY_COLLISIONS").sum().alias("injury_collisions"),
        pl.col("FTR_COLLISIONS").sum().alias("ftr_collisions"),
        pl.col("PD_COLLISIONS").sum().alias("pd_collisions"),
        pl.col("PEDESTRIAN").sum().alias("pedestrian_involved"),
        pl.col("BICYCLE").sum().alias("cyclist_involved"),
    )
    .sort("year_month")
)

ym_ts_order = by_month_ts["year_month"].to_list()

with pl.Config(tbl_rows=5):
    display(by_month_ts)

year_month,total_collisions,fatalities,injury_collisions,ftr_collisions,pd_collisions,pedestrian_involved,cyclist_involved
str,u32,i32,u32,u32,u32,u32,u32
"""2014-01""",6138,2,695,996,4469,153,14
"""2014-02""",5791,3,627,1012,4189,143,12
"""2014-03""",5275,1,650,1011,3644,152,27
…,…,…,…,…,…,…,…
"""2025-11""",5765,2,884,1135,3756,199,87
"""2025-12""",6137,4,798,1186,4160,174,33


In [6]:
base_ts = alt.Chart(by_month_ts).encode(
    x=alt.X(
        "year_month:N",
        sort=ym_ts_order,
        title="Month",
        axis=alt.Axis(labelAngle=-60, labelOverlap=True, tickCount=24),
    ),
)

chart_total = (
    base_ts.mark_line(color="#4C72B0")
    .encode(
        y=alt.Y("total_collisions:Q", title="Total Collisions"),
        tooltip=["year_month", "total_collisions"],
    )
    .properties(title="Total Collisions per Month", width=800, height=180)
)

chart_fatal = (
    base_ts.mark_line(color="#C44E52")
    .encode(
        y=alt.Y("fatalities:Q", title="Fatalities"),
        tooltip=["year_month", "fatalities"],
    )
    .properties(title="Fatalities per Month", width=800, height=180)
)

chart_injury = (
    base_ts.mark_line(color="#DD8452")
    .encode(
        y=alt.Y("injury_collisions:Q", title="Injury Collisions"),
        tooltip=["year_month", "injury_collisions"],
    )
    .properties(title="Injury Collisions per Month", width=800, height=180)
)

alt.vconcat(chart_total, chart_fatal, chart_injury).properties(spacing=10)

alt.VConcatChart(...)

## 3. Seasonality — Collisions by Month

In [7]:
MONTH_ORDER = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

by_month = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .group_by("OCC_MONTH")
    .agg(
        pl.len().alias("total_collisions"),
        pl.col("FATALITIES").sum().alias("fatalities"),
        pl.col("INJURY_COLLISIONS").sum().alias("injury_collisions"),
        pl.col("PEDESTRIAN").sum().alias("pedestrian_involved"),
        pl.col("BICYCLE").sum().alias("cyclist_involved"),
    )
)

by_month = by_month.with_columns(
    pl.col("OCC_MONTH")
    .map_elements(lambda m: MONTH_ORDER.index(m), return_dtype=pl.Int32)
    .alias("month_idx")
).sort("month_idx")

base_month = alt.Chart(by_month).encode(
    x=alt.X("OCC_MONTH:N", sort=MONTH_ORDER, title="Month"),
)

chart_m_total = (
    base_month.mark_bar(color="#4C72B0")
    .encode(
        y=alt.Y("total_collisions:Q", title="Total Collisions"),
        tooltip=["OCC_MONTH", "total_collisions"],
    )
    .properties(title="Total Collisions by Month (all years)", width=700, height=160)
)

chart_m_ped = (
    base_month.mark_bar(color="#8172B3")
    .encode(
        y=alt.Y("pedestrian_involved:Q", title="Pedestrian Involved"),
        tooltip=["OCC_MONTH", "pedestrian_involved"],
    )
    .properties(title="Pedestrian-Involved Collisions by Month", width=700, height=160)
)

chart_m_cyc = (
    base_month.mark_bar(color="#55A868")
    .encode(
        y=alt.Y("cyclist_involved:Q", title="Cyclist Involved"),
        tooltip=["OCC_MONTH", "cyclist_involved"],
    )
    .properties(title="Cyclist-Involved Collisions by Month", width=700, height=160)
)

alt.vconcat(chart_m_total, chart_m_ped, chart_m_cyc).properties(spacing=10)

alt.VConcatChart(...)

## 4. Day-of-Week Patterns

In [8]:
DOW_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

by_dow = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .group_by("OCC_DOW")
    .agg(
        pl.len().alias("total_collisions"),
        pl.col("FATALITIES").sum().alias("fatalities"),
        pl.col("INJURY_COLLISIONS").sum().alias("injury_collisions"),
    )
)

base_dow = alt.Chart(by_dow).encode(
    x=alt.X("OCC_DOW:N", sort=DOW_ORDER, title="Day of Week"),
)

c1 = (
    base_dow.mark_bar(color="#4C72B0")
    .encode(
        y=alt.Y("total_collisions:Q", title="Total"),
        tooltip=["OCC_DOW", "total_collisions"],
    )
    .properties(title="Collisions by Day of Week", width=400, height=200)
)

c2 = (
    base_dow.mark_bar(color="#C44E52")
    .encode(
        y=alt.Y("fatalities:Q", title="Fatalities"),
        tooltip=["OCC_DOW", "fatalities"],
    )
    .properties(title="Fatalities by Day of Week", width=400, height=200)
)

c1 | c2

alt.HConcatChart(...)

## 5. Hour of Day — When Do Collisions Happen?

In [9]:
by_hour = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .group_by("OCC_HOUR")
    .agg(
        pl.len().alias("total_collisions"),
        pl.col("FATALITIES").sum().alias("fatalities"),
        pl.col("INJURY_COLLISIONS").sum().alias("injury_collisions"),
        pl.col("PEDESTRIAN").sum().alias("pedestrian_involved"),
    )
    .sort("OCC_HOUR")
    .with_columns(pl.col("OCC_HOUR").cast(pl.Utf8))
)

hour_order = [str(h) for h in range(24)]

base_h = alt.Chart(by_hour).encode(
    x=alt.X("OCC_HOUR:N", sort=hour_order, title="Hour of Day (0 = midnight)"),
)

h1 = (
    base_h.mark_bar(color="#4C72B0")
    .encode(
        y=alt.Y("total_collisions:Q", title="Collisions"),
        tooltip=["OCC_HOUR", "total_collisions"],
    )
    .properties(title="Total Collisions by Hour", width=700, height=160)
)

h2 = (
    base_h.mark_bar(color="#C44E52")
    .encode(
        y=alt.Y("fatalities:Q", title="Fatalities"),
        tooltip=["OCC_HOUR", "fatalities"],
    )
    .properties(
        title="Fatalities by Hour (late night proportionally deadlier)", width=700, height=160
    )
)

h3 = (
    base_h.mark_bar(color="#8172B3")
    .encode(
        y=alt.Y("pedestrian_involved:Q", title="Pedestrian Involved"),
        tooltip=["OCC_HOUR", "pedestrian_involved"],
    )
    .properties(title="Pedestrian-Involved Collisions by Hour", width=700, height=160)
)

alt.vconcat(h1, h2, h3).properties(spacing=10)

alt.VConcatChart(...)

## 6. Collision Severity Breakdown

In [10]:
severity = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .with_columns(
        pl.when(pl.col("FATALITIES") > 0)
        .then(pl.lit("Fatal"))
        .when(pl.col("INJURY_COLLISIONS"))
        .then(pl.lit("Injury"))
        .when(pl.col("FTR_COLLISIONS"))
        .then(pl.lit("Fail to Remain"))
        .otherwise(pl.lit("Property Damage Only"))
        .alias("severity")
    )
    .group_by("severity")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum() * 100).round(1).alias("pct"))
    .sort("count", descending=True)
)
display(severity)

alt.Chart(severity).mark_bar().encode(
    x=alt.X("count:Q", title="Collisions"),
    y=alt.Y("severity:N", sort="-x", title=""),
    color=alt.Color(
        "severity:N",
        legend=None,
        scale=alt.Scale(
            domain=["Fatal", "Injury", "Fail to Remain", "Property Damage Only"],
            range=["#C44E52", "#DD8452", "#8172B3", "#4C72B0"],
        ),
    ),
    tooltip=["severity", "count", alt.Tooltip("pct:Q", title="%", format=".1f")],
).properties(title="Collision Severity (all years)", width=600, height=200)

severity,count,pct
str,u32,f64
"""Property Damage Only""",550789,69.7
"""Fail to Remain""",131819,16.7
"""Injury""",107497,13.6
"""Fatal""",658,0.1


alt.Chart(...)

## 7. Vulnerable Road Users — Pedestrians & Cyclists Over Time

In [11]:
vru_year = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .group_by("OCC_YEAR")
    .agg(
        pl.len().alias("total"),
        pl.col("PEDESTRIAN").sum().alias("pedestrian"),
        pl.col("BICYCLE").sum().alias("cyclist"),
        pl.col("MOTORCYCLE").sum().alias("motorcycle"),
    )
    .sort("OCC_YEAR")
    .with_columns(pl.col("OCC_YEAR").cast(pl.Utf8))
)

vru_long = vru_year.unpivot(
    on=["pedestrian", "cyclist", "motorcycle"],
    index="OCC_YEAR",
    variable_name="road_user",
    value_name="collisions",
)

year_order = vru_year["OCC_YEAR"].to_list()

alt.Chart(vru_long).mark_line(point=True).encode(
    x=alt.X("OCC_YEAR:N", sort=year_order, title="Year"),
    y=alt.Y("collisions:Q", title="Collisions"),
    color=alt.Color(
        "road_user:N",
        title="Road User",
        scale=alt.Scale(
            domain=["pedestrian", "cyclist", "motorcycle"], range=["#8172B3", "#55A868", "#DD8452"]
        ),
    ),
    tooltip=["OCC_YEAR", "road_user", "collisions"],
).properties(title="Vulnerable Road User Collisions per Year", width=700, height=280)

alt.Chart(...)

## 8. COVID Impact — 2020–2021 Drop

In [12]:
covid_ts = by_month_ts.filter(pl.col("year_month").str.slice(0, 4).cast(pl.Int32).is_between(2018, 2023))
covid_order = covid_ts["year_month"].to_list()

alt.Chart(covid_ts).mark_line(color="#4C72B0").encode(
    x=alt.X(
        "year_month:N",
        sort=covid_order,
        title="Month",
        axis=alt.Axis(labelAngle=-60, labelOverlap=True),
    ),
    y=alt.Y("total_collisions:Q", title="Collisions"),
    tooltip=["year_month", "total_collisions"],
).properties(
    title="Monthly Collisions 2018-2023 (COVID lockdowns visible in 2020-2021)",
    width=800,
    height=260,
)

alt.Chart(...)

## 9. Top Neighbourhoods by Collision Count

In [13]:
top_hoods = (
    df.filter(
        pl.col("OCC_YEAR") < 2026,
        pl.col("NEIGHBOURHOOD_158").is_not_null(),
    )
    .group_by("NEIGHBOURHOOD_158")
    .agg(
        pl.len().alias("total_collisions"),
        pl.col("FATALITIES").sum().alias("fatalities"),
        pl.col("INJURY_COLLISIONS").sum().alias("injuries"),
        pl.col("PEDESTRIAN").sum().alias("pedestrian"),
    )
    .sort("total_collisions", descending=True)
    .head(20)
)

alt.Chart(top_hoods).mark_bar(color="#4C72B0").encode(
    x=alt.X("total_collisions:Q", title="Total Collisions"),
    y=alt.Y("NEIGHBOURHOOD_158:N", sort="-x", title=""),
    tooltip=["NEIGHBOURHOOD_158", "total_collisions", "fatalities", "injuries", "pedestrian"],
).properties(title="Top 20 Neighbourhoods by Total Collisions (2014-2025)", width=600, height=420)

alt.Chart(...)

## 10. Fatality Rate by Neighbourhood (min 500 collisions)

In [14]:
hood_fatal = (
    df.filter(
        pl.col("OCC_YEAR") < 2026,
        pl.col("NEIGHBOURHOOD_158").is_not_null(),
    )
    .group_by("NEIGHBOURHOOD_158")
    .agg(
        pl.len().alias("total"),
        pl.col("FATALITIES").sum().alias("fatalities"),
    )
    .filter(pl.col("total") >= 500)
    .with_columns(
        (pl.col("fatalities") / pl.col("total") * 1000).round(2).alias("fatality_rate_per_1k")
    )
    .sort("fatality_rate_per_1k", descending=True)
    .head(20)
)

alt.Chart(hood_fatal).mark_bar(color="#C44E52").encode(
    x=alt.X("fatality_rate_per_1k:Q", title="Fatalities per 1 000 Collisions"),
    y=alt.Y("NEIGHBOURHOOD_158:N", sort="-x", title=""),
    tooltip=[
        "NEIGHBOURHOOD_158",
        alt.Tooltip("fatality_rate_per_1k:Q", title="Rate per 1k"),
        "fatalities",
        "total",
    ],
).properties(
    title="Top 20 Neighbourhoods by Fatality Rate (per 1 000 collisions, >=500 collisions)",
    width=600,
    height=420,
)

alt.Chart(...)

## 11. Fail-to-Remain (Hit & Run) Trend

In [15]:
ftr_year = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .group_by("OCC_YEAR")
    .agg(
        pl.len().alias("total"),
        pl.col("FTR_COLLISIONS").sum().alias("ftr"),
    )
    .sort("OCC_YEAR")
    .with_columns(
        (pl.col("ftr") / pl.col("total") * 100).round(2).alias("ftr_pct"),
        pl.col("OCC_YEAR").cast(pl.Utf8),
    )
)

year_ord = ftr_year["OCC_YEAR"].to_list()

bar = (
    alt.Chart(ftr_year)
    .mark_bar(color="#8172B3")
    .encode(
        x=alt.X("OCC_YEAR:N", sort=year_ord, title="Year"),
        y=alt.Y("ftr:Q", title="Hit & Run Collisions"),
        tooltip=["OCC_YEAR", "ftr", alt.Tooltip("ftr_pct:Q", title="% of all", format=".1f")],
    )
    .properties(title="Fail-to-Remain (Hit & Run) Collisions per Year", width=700, height=200)
)

line = (
    alt.Chart(ftr_year)
    .mark_line(color="#C44E52", strokeDash=[4, 2], point=True)
    .encode(
        x=alt.X("OCC_YEAR:N", sort=year_ord),
        y=alt.Y("ftr_pct:Q", title="Hit & Run as % of All Collisions"),
        tooltip=["OCC_YEAR", alt.Tooltip("ftr_pct:Q", title="%", format=".1f")],
    )
    .properties(title="Hit & Run as % of All Collisions", width=700, height=200)
)

alt.vconcat(bar, line).properties(spacing=10)

alt.VConcatChart(...)

## 12. Heatmap — Hour × Day of Week

In [16]:
heatmap_data = (
    df.filter(pl.col("OCC_YEAR") < 2026)
    .group_by(["OCC_DOW", "OCC_HOUR"])
    .agg(pl.len().alias("collisions"))
    .with_columns(pl.col("OCC_HOUR").cast(pl.Utf8))
)

alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X("OCC_HOUR:N", sort=hour_order, title="Hour of Day"),
    y=alt.Y("OCC_DOW:N", sort=DOW_ORDER, title="Day of Week"),
    color=alt.Color("collisions:Q", title="Collisions", scale=alt.Scale(scheme="blues")),
    tooltip=["OCC_DOW", "OCC_HOUR", "collisions"],
).properties(title="Collision Heatmap - Hour of Day x Day of Week", width=700, height=220)

alt.Chart(...)